<a href="https://colab.research.google.com/github/lshofsl/Thesis/blob/main/GeneNCA_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **EngramNCA**

In [ ]:
#@title Get Packages and Images  { vertical-output: true}

import os
import sys

!rm -rf Thesis
!git clone https://github.com/lshofsl/Thesis

repo_path = os.path.abspath("Thesis")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if os.path.exists(repo_path):
    print("Success! Files now found:", os.listdir(repo_path))

In [ ]:
#@title Imports { vertical-output: true}
import random
import numpy as np
import matplotlib.pyplot as plt
import os
import torch

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

from NCA.NCA_gene import *
import NCA.utils as utils
from IPython.display import Image, HTML, clear_output
import torch.nn.functional as F
import logging
from IPython.display import display, HTML, Video
from PIL import Image
import cv2
from base64 import b64encode
logger = logging.getLogger()
old_level = logger.level
logger.setLevel(100)




In [ ]:
#@title Setup { vertical-output: true}
HEIGHT = 30 #@param {type:"integer"}
WIDTH = 30 #@param {type:"integer"}
CHANNELS = 22 # @param {type:"integer"}<--- NCA feature channels
BATCH_SIZE = 12 #@param {type:"integer"}
PADDING = 5 #@param {type:"integer"}
GENE_COUNT = 3 #@param {type:"integer"} <-- Number of gene channels to use for "private" information
RECURRENT_CHANNELS = 3 #@param {type:"integer"} <-- Number of channels for RA computation, these channels are not updated by the NCA and are only used for recurrent processing
MODULATORY_OUTPUT = 3 #@param {type:"integer"} <-- Number of channels for modulatory output, these channels are not updated  and only show the output of the NCA
POOL_SIZE = 2666 #@param {type:"integer"}<--- NCA training pool size, lower values train faster but are less stable
TRAINING_ITERS = 10000  #@param {type:"integer"}<-- Number of trainign iterations
HIDDEN_SIZE = 64 #@param {type:"integer"}<--- NCA hidden size
PRIMITIVES_SHAPES = ["Images/square.png", "Images/circle.png", "Images/triangle.png"]
PRIMITIVES_BODY_PARTS = ["Images/Torso.png", "Images/Head.png", "Images/Tail.png"] # "Images/leg1.png", "Images/leg2.png", "Images/leg3.png", 'Images/leg4.png']
PRIMITIVES_LINES = ["Images/horizontal.png", "Images/Verical.png"]
style = """
<style>
.output_wrapper, .output {
    display: flex;
    flex-direction: row-reverse; /* Align content to the right */
}
</style>
"""


In [ ]:
#@title Load Primitives { vertical-output: true}


paths = [os.path.join(repo_path, p) for p in PRIMITIVES_BODY_PARTS]

images = []
images_to_display = []
for path in paths:
    image, image_to_display = utils.get_image(path, HEIGHT, WIDTH, padding=PADDING)
    images.append(image)
    images_to_display.append(image_to_display)

genes = [[0], [1], [2]] # <-- Gene one hot encoding, indicates which bits if the gene sequence for each encoded "image" should be 1, [0] = 001, [0,1] = 011, [2] = 100 etc. for 3 bits genes. One, one-hot encoding per image, this rule applies for any gene size

HEIGHT = HEIGHT + 2*PADDING
WIDTH = WIDTH + 2*PADDING
assert len(paths) == len(genes), 'Genes and images should have the same length '

In [ ]:
#@title Display Primitives { vertical-output: true}
for i,image in enumerate(images_to_display):
    plt.figure(3+i)
    plt.imshow(image)
pools = []
for gene in genes:
    pools.append(utils.make_gene_pool_GeneCA(gene, pool_size=POOL_SIZE,height=HEIGHT, width=WIDTH, channels=CHANNELS, gene_size=GENE_COUNT, device=DEVICE))
seeds = []
for pool in pools:
    seeds.append(pool[0].clone())

In [ ]:
#@title Get Batch Image Partitions { vertical-output: true}
partitions = len(paths)
if partitions == 1:
    part = [BATCH_SIZE]
div = BATCH_SIZE//partitions
rem = BATCH_SIZE % partitions
part = [div + 1 if i < rem else div for i in range(partitions)]
print(f"Batch image paritions = {part}. Batch Size of {BATCH_SIZE}. Number of Partitions = {partitions}")

In [ ]:
#@title Load Filters for Loss Function { vertical-output: true}
sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=torch.float32, device=DEVICE)
lap = torch.tensor([[1.0, 2.0, 1.0], [2.0, -12, 2.0], [1.0, 2.0, 1.0]], dtype=torch.float32, device=DEVICE)
filters = torch.stack([sobel_x, sobel_x.T, lap])
folder = "Gene"

In [ ]:
#@title Create Path for Saving Models { vertical-output: true}
path = "Trained_models/" + folder
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Path: {path} created")
else:
    print(f"Path: {path} already exists, all OK!")


In [ ]:
#@title Initialise NCA { vertical-output: true}
bases = [images[i].tile(part[i],1,1,1) for i in range(len(part))]
base = torch.cat(bases, dim =0 ).to(DEVICE)
loss_log = []
nca = GeneCA(CHANNELS,HIDDEN_SIZE, gene_size=GENE_COUNT)
nca = nca.to(DEVICE)
optim = torch.optim.AdamW(nca.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=2000, gamma=0.3)
name = folder + "/" +type(nca).__name__ + "_gene_size_" +str(GENE_COUNT)

In [ ]:
#@title Training { vertical-output: true}

lambda_p = 0.1
lambda_m = 0.1

alpha_log =  []
beta_log   = []
omega_log  = []
K_log      = []
kappa_log  = []
log_iters  = []
a          = []
b          = []
d          = []
first_hiddens = []

for i in range(TRAINING_ITERS + 1):
    with torch.no_grad():
        idxs, x = utils.get_gene_pool(pools, part, seeds)

    # Ensure x is ready for gradients
    x = x.detach().clone()
    x.requires_grad = True

    optim.zero_grad()

    # Initialize loss tensors on the correct device
    total_amp_loss = torch.tensor(0.0, device=x.device)
    total_smooth_loss = torch.tensor(0.0, device=x.device)

    # Use a local list to track states if needed, but avoid modifying 'x' in-place
    current_x = x

    itters = random.randrange(32, 92)
    for t in range(itters):
        m_prev = current_x[:, 19:22].clone()

        out, phase, amplitud = nca(current_x, step=t, k=4)
        current_x = out

        total_amp_loss += amplitud.pow(2).mean()
        total_smooth_loss += (current_x[:, 19:22] - m_prev).pow(2).mean()

    # Image loss using the final state
    x_rgb = current_x[:, :4, :, :]
    loss_img = (base - x_rgb).pow(2).mean() + \
               0.1 * (perchannel_conv(base, filters) - perchannel_conv(x_rgb, filters)).pow(2).mean()

    loss = loss_img + (lambda_p * total_amp_loss / itters) + (lambda_m * total_smooth_loss / itters)

    loss.backward()

    # Gradient clipping (Stabilizes NCA training)
    for p in nca.parameters():
        if p.grad is not None:
            p.grad /= (p.grad.norm() + 1e-8)

    optim.step()

    # 6. Housekeeping (Use no_grad for memory management)
    loss_log.append(loss.log().item())
    with torch.no_grad():
        # Update pool with detached state so we don't carry the graph
        pools = utils.udate_gene_pool(pools, current_x.detach(), idxs, part)

    scheduler.step()

    if i % 100 == 0:
        print(f"Training itter {i}, loss = {loss.item()}")
        plt.clf()
        clear_output()
        plt.figure(1,figsize=(10, 4))
        plt.title('Loss history')
        print(name)
        plt.plot(loss_log, '.', alpha=0.5, color = "b")
        utils.show_batch(x[2:10])
        plt.show(block=False)
        plt.pause(0.01)

        alpha_log.append(nca.alpha.item())
        beta_log.append(nca.beta.item())
        omega_log.append(nca.omega.item())
        K_log.append(nca.K.item())
        kappa_log.append(nca.kappa.item())
        log_iters.append(i)
        a.append(current_x[:, 16:17].detach().cpu().numpy())
        b.append(current_x[:, 17:18].detach().cpu().numpy())
        d.append(current_x[:, 18:19].detach().cpu().numpy())
        first_hiddens.append(current_x[:, 4:8].detach().cpu().numpy())


    if i % 100 == 0:
        torch.save(nca.state_dict(), "Trained_models/" + name + ".pth")

In [ ]:
plt.plot(log_iters, alpha_log, label='alpha')
plt.plot(log_iters, beta_log, label='beta')
plt.plot(log_iters, omega_log, label='omega')
plt.plot(log_iters, K_log, label='K')
plt.legend()
plt.show()



In [ ]:
a = np.array(a)
b = np.array(b)
d = np.array(d)
first_hiddens = np.array(first_hiddens)

In [ ]:
a.shape

In [ ]:
import matplotlib.animation as animation
# 1. Reshape the 12 dim into (3 morphologies, 4 samples)
# New shape: (81, 3, 4, 1, 50, 50)
reshaped_data = a.reshape(101, 3, 4, 1, 40, 40)

# Let's select Sample Index 0 for all 3 morphologies
# We also drop the 1-channel dimension for plotting -> shape: (81, 3, 50, 50)
sample_idx = 0
grid_data = reshaped_data[:, :, sample_idx, 0, :, :]

# 2. Set up a multi-plot figure (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
morphology_names = ["Morphology A", "Morphology B", "Morphology C"]
ims = []

# Initialize the 3 plots
for i, ax in enumerate(axes):
    im = ax.imshow(grid_data[0, i], cmap='bwr', vmin=grid_data.min(), vmax=grid_data.max())
    ax.set_title(morphology_names[i])
    ax.axis('off') # Hide pixel grid axes for a cleaner look
    ims.append(im)

# 3. Animation update function
def update(t_idx):
    actual_frame = log_iters[t_idx]
    fig.suptitle(f"Simulation Time: Frame {actual_frame}", fontsize=14)

    # Update all 3 images for the current time step
    for i, im in enumerate(ims):
        im.set_array(grid_data[t_idx, i])
    return ims

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=len(log_iters), interval=100, blit=False)

# 4. Save the side-by-side comparison
ani.save('morphology_comparison_a.gif', writer='pillow', fps=10)

In [ ]:
reshaped_data = b.reshape(101, 3, 4, 1, 40, 40)

# Let's select Sample Index 0 for all 3 morphologies
# We also drop the 1-channel dimension for plotting -> shape: (81, 3, 50, 50)
sample_idx = 0
grid_data = reshaped_data[:, :, sample_idx, 0, :, :]

# 2. Set up a multi-plot figure (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
morphology_names = ["Morphology A", "Morphology B", "Morphology C"]
ims = []

# Initialize the 3 plots
for i, ax in enumerate(axes):
    im = ax.imshow(grid_data[0, i], cmap='bwr', vmin=grid_data.min(), vmax=grid_data.max())
    ax.set_title(morphology_names[i])
    ax.axis('off') # Hide pixel grid axes for a cleaner look
    ims.append(im)

# 3. Animation update function
def update(t_idx):
    actual_frame = log_iters[t_idx]
    fig.suptitle(f"Simulation Time: Frame {actual_frame}", fontsize=14)

    # Update all 3 images for the current time step
    for i, im in enumerate(ims):
        im.set_array(grid_data[t_idx, i])
    return ims

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=len(log_iters), interval=100, blit=False)

# 4. Save the side-by-side comparison
ani.save('morphology_comparison_b.gif', writer='pillow', fps=10)

In [ ]:
reshaped_data = d.reshape(101, 3, 4, 1, 40, 40)

# Let's select Sample Index 0 for all 3 morphologies
# We also drop the 1-channel dimension for plotting -> shape: (81, 3, 50, 50)
sample_idx = 0
grid_data = reshaped_data[:, :, sample_idx, 0, :, :]

# 2. Set up a multi-plot figure (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
morphology_names = ["Morphology A", "Morphology B", "Morphology C"]
ims = []

# Initialize the 3 plots
for i, ax in enumerate(axes):
    im = ax.imshow(grid_data[0, i], cmap='YlOrRd', vmin=grid_data.min(), vmax=grid_data.max())
    ax.set_title(morphology_names[i])
    ax.axis('off') # Hide pixel grid axes for a cleaner look
    ims.append(im)

# 3. Animation update function
def update(t_idx):
    actual_frame = log_iters[t_idx]
    fig.suptitle(f"Simulation Time: Frame {actual_frame}", fontsize=14)

    # Update all 3 images for the current time step
    for i, im in enumerate(ims):
        im.set_array(grid_data[t_idx, i])
    return ims

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=len(log_iters), interval=100, blit=False)

# 4. Save the side-by-side comparison
ani.save('morphology_comparison_d.gif', writer='pillow', fps=10)

In [ ]:
nca = GeneCA(CHANNELS,hidden_n=HIDDEN_SIZE, gene_size=GENE_COUNT)
nca.load_state_dict(torch.load("Trained_models/Gene/GeneCA_gene_size_3.pth"))
nca.to(DEVICE).eval()

In [ ]:
with torch.no_grad():
    # Start from a pool sample — guaranteed good starting state
    x_test = pools[2][0:1].clone().to(DEVICE)

    # Grow for 100 steps
    for t in range(500):
        x_test, phase, amp = nca(x_test, step=t, k=4)

    # --- Plot 1: What it looks like after growth ---
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(x_test[0, :4].permute(1,2,0).cpu().clip(0,1))
    axes[0].set_title('RGBA after 100 steps')
    axes[1].imshow(x_test[0, 15].cpu(), cmap='RdBu')
    axes[1].set_title('RA: a')
    axes[2].imshow(x_test[0, 16].cpu(), cmap='RdBu')
    axes[2].set_title('RA: b')
    axes[3].imshow(x_test[0, 17].cpu(), cmap='hot')
    axes[3].set_title('RA: d (injury field)')
    plt.suptitle('RA spatial organization after growth')
    plt.tight_layout()
    plt.show()

    # --- Plot 2: Damage experiment ---
    x_damaged = x_test.clone()
    x_damaged[:, :, :, x_damaged.shape[3]//2:] = 0  # remove right half

    amps, losses = [], []
    frames_damaged = [x_damaged[0, :4].permute(1,2,0).cpu().clip(0,1)]

    for t in range(200):
        x_damaged, phase, amp = nca(x_damaged, step=t, k=4)
        amps.append(amp.mean().item())
        losses.append((base[0:1] - x_damaged[:, :4]).pow(2).mean().item())
        if t in [25, 50, 100, 150, 199]:
            frames_damaged.append(
                x_damaged[0, :4].permute(1,2,0).cpu().clip(0,1)
            )

    # Show regeneration frames
    fig, axes = plt.subplots(1, len(frames_damaged), figsize=(20, 4))
    titles = ['damage'] + [f't={t}' for t in [25, 50, 100, 150, 199]]
    for ax, frame, title in zip(axes, frames_damaged, titles):
        ax.imshow(frame)
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('Regeneration after damage (right half removed)')
    plt.tight_layout()
    plt.show()

    # --- Plot 3: RA amplitude and loss over regeneration ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
    ax1.plot(amps)
    ax1.axvline(x=0, color='r', linestyle='--', label='damage applied')
    ax1.set_title('RA mean amplitude ρ during regeneration')
    ax1.set_ylabel('mean ρ')
    ax1.legend()
    ax2.plot(losses)
    ax2.set_title('Reconstruction loss during regeneration')
    ax2.set_ylabel('MSE')
    ax2.set_xlabel('step after damage')
    plt.tight_layout()
    plt.show()



In [ ]:
del nca
del optim
torch.cuda.empty_cache()